# AI Detector Results Analysis

This notebook analyzes previously collected AI-detector results; it does not run detectors. It reads Excel workbooks from Google Drive and produces reports, summary statistics, plots, and files for manual review.

## Analyses

1. **Calibration on reference texts**: evaluates detectors using `AI-Free`, `Humanized`, and `AI-Full` samples to support detector weighting.
2. **Assignment-level analysis**: summarizes the assignments dataset, calculates a weighted combined AI rate, compares university and language groups, generates plots, and flags files with substantial detector disagreement.
3. **Conflict analysis**: calculates per-file disagreement metrics and compares conflicts by language.

## Files

**Inputs:** Excel workbooks containing detector results.

**Outputs:** TXT reports, CSV summaries, PNG plots, and copies of raw PDFs for manual review of conflicting cases.

Each major code section identifies its purpose, expected inputs, outputs, and role in the workflow.

## Detector calibration

Evaluate each detector on a controlled sample with known labels:

- **AI-Free** should be classified as human.
- **Humanized** should remain classified as AI-derived.
- **AI-Full** should be classified as AI.

Per-tool, per-set accuracy is used to weight detectors in the combined AI-rate for the full assignment dataset. The next code block mounts Google Drive, loads the testing-sample workbook, standardizes detector labels, calculates accuracy and mean AI rates, and writes a TXT summary.

In [ ]:
import os
from pathlib import Path

import pandas as pd

XLSX_PATH = Path(os.environ["AI_DETECTION_WORKBOOK"])
TXT_OUTPUT_PATH = Path(os.environ["AI_DETECTION_REPORT"])

TOOLS = ["gptzero", "pangram", "sapling", "isgen"]
EXPECTED = {
    "ai_free": "HUMAN",
    "humanized": "AI",
    "ai_full": "AI",
}
SET_LABELS = {
    "ai_free": "AI-Free",
    "humanized": "Humanized",
    "ai_full": "AI-Full",
}
TOOL_LABELS = {
    "gptzero": "GPTZero",
    "pangram": "Pangram",
    "sapling": "Sapling",
    "isgen": "Isgen",
}


def normalize_label(value):
    """Map detector output to a common label."""
    if pd.isna(value):
        return "UNKNOWN"

    label = str(value).strip().lower()
    if label in {"human", "likely human"}:
        return "HUMAN"
    if label in {"ai", "likely ai"}:
        return "AI"
    if label == "mixed":
        return "MIXED"
    if label == "error":
        return "ERROR"
    if "mixed" in label:
        return "MIXED"
    if "error" in label:
        return "ERROR"
    if "human" in label and "ai" not in label:
        return "HUMAN"
    if "ai" in label:
        return "AI"
    return "UNKNOWN"


def pct(numerator, denominator):
    return round(100 * numerator / denominator, 2) if denominator else 0.0


def safe_mean(series):
    values = pd.to_numeric(series, errors="coerce").dropna()
    return round(float(values.mean()), 2) if len(values) else None


if not XLSX_PATH.is_file():
    raise FileNotFoundError(f"Workbook not found: {XLSX_PATH}")

df = pd.read_excel(XLSX_PATH)

required_cols = ["file_name"]
for version in EXPECTED:
    for tool in TOOLS:
        required_cols.extend(
            [
                f"{version}_{tool}_main_result",
                f"{version}_{tool}_ai_rate_percent",
            ]
        )

missing = [column for column in required_cols if column not in df.columns]
if missing:
    raise ValueError("Missing required columns:\n" + "\n".join(missing))

report_lines = [
    "=" * 78,
    "AI-DETECTION ACCURACY REPORT",
    "=" * 78,
    f"Input file: {XLSX_PATH}",
    f"Number of base files tested: {len(df)}",
    "",
    "Expected truth labels:",
    "  - AI-Free   = HUMAN",
    "  - Humanized = AI",
    "  - AI-Full   = AI",
    "",
]
overall_rows = []

for tool in TOOLS:
    report_lines.extend(["-" * 78, TOOL_LABELS[tool].upper(), "-" * 78])
    total_correct = 0
    total_cases = 0

    for version, expected in EXPECTED.items():
        result_col = f"{version}_{tool}_main_result"
        rate_col = f"{version}_{tool}_ai_rate_percent"
        predicted = df[result_col].apply(normalize_label)
        correct = int((predicted == expected).sum())
        total = len(df)
        accuracy = pct(correct, total)
        avg_ai_rate = safe_mean(df[rate_col])
        counts = predicted.value_counts(dropna=False).to_dict()

        report_lines.append(
            f"{SET_LABELS[version]:10s} | Accuracy: {correct}/{total} = {accuracy:.2f}% | "
            f"Avg AI rate: {avg_ai_rate if avg_ai_rate is not None else 'N/A'}%"
        )
        report_lines.append(
            f"{'':10s} | Predictions -> HUMAN: {counts.get('HUMAN', 0)}, "
            f"AI: {counts.get('AI', 0)}, MIXED: {counts.get('MIXED', 0)}, "
            f"ERROR: {counts.get('ERROR', 0)}, UNKNOWN: {counts.get('UNKNOWN', 0)}"
        )

        total_correct += correct
        total_cases += total
        overall_rows.append(
            {
                "tool": TOOL_LABELS[tool],
                "set": SET_LABELS[version],
                "correct": correct,
                "total": total,
                "accuracy_percent": accuracy,
                "avg_ai_rate_percent": avg_ai_rate,
            }
        )

    report_lines.extend(
        [
            "",
            f"OVERALL for {TOOL_LABELS[tool]}: {total_correct}/{total_cases} = "
            f"{pct(total_correct, total_cases):.2f}%",
            "",
        ]
    )

summary_df = pd.DataFrame(overall_rows)
pivot_acc = summary_df.pivot(index="tool", columns="set", values="accuracy_percent")
pivot_rate = summary_df.pivot(index="tool", columns="set", values="avg_ai_rate_percent")

report_lines.extend(
    [
        "=" * 78,
        "COMPACT ACCURACY SUMMARY (%)",
        "=" * 78,
        pivot_acc.to_string(),
        "",
        "=" * 78,
        "COMPACT AVERAGE AI-RATE SUMMARY (%)",
        "=" * 78,
        pivot_rate.to_string(),
        "",
    ]
)

report_text = "\n".join(report_lines)
TXT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
TXT_OUTPUT_PATH.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"\nSaved TXT report to:\n{TXT_OUTPUT_PATH}")

All four detectors classified the clearly human-written AI-Free and clearly AI-generated AI-Full files well, but performance declined substantially on Humanized files. Post-editing or humanization therefore made AI-origin text harder to detect.

Across the three reference sets, GPTZero performed best overall, followed by Sapling and Pangram; Isgen performed worst. Because all tools were weakest on Humanized text, later analyses use weighted detector contributions rather than an equal-weight average and interpret results cautiously when text may have been revised after AI generation.

## 2. Assignment-Level Detector Summary

This analysis uses the full processed assignment dataset, not the controlled testing sample.

For written assignments, the combined AI-rate is the weighted mean of available detector rates. For coding assignments, it uses Pangram only because Pangram is the only detector available in that workflow branch.

The following code loads the assignment-level results workbook; computes combined rates; summarizes results by university, year, major, language, assignment type, and other groups; runs correlation and group-difference analyses; generates plots; identifies files with substantial detector disagreement; exports those conflicts to CSV; and copies the corresponding raw PDFs to a review folder.

In [ ]:
# Install dependencies when running in a fresh Colab runtime.
!pip -q install openpyxl scipy statsmodels

import os
import re
import shutil
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import kruskal, pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    pass

warnings.filterwarnings("ignore")

# Configure these paths through environment variables or edit the defaults.
INPUT_XLSX = os.environ.get("AI_DETECTION_INPUT_XLSX", "analysis_input.xlsx")
OUTPUT_DIR = os.environ.get("AI_DETECTION_OUTPUT_DIR", "analysis_output")
ASSIGNMENTS_HUB = os.environ.get("AI_DETECTION_PDF_ROOT", "raw_pdfs")

REPORT_TXT = os.path.join(OUTPUT_DIR, "Assignments_AI_Analysis_Report.txt")
PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
CONFLICT_PDFS_DIR = os.path.join(OUTPUT_DIR, "Conflicting_Raw_PDFs")
CONFLICT_CSV = os.path.join(OUTPUT_DIR, "conflicting_files.csv")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)
os.makedirs(CONFLICT_PDFS_DIR, exist_ok=True)

# Detector weights are based on measured accuracy in the validation sample.
RAW_WEIGHTS = {
    "gptzero": 93.33,
    "pangram": 84.00,
    "sapling": 88.00,
    "isgen": 72.67,
}
weight_sum = sum(RAW_WEIGHTS.values())
NORM_WEIGHTS = {tool: weight / weight_sum for tool, weight in RAW_WEIGHTS.items()}

TOOLS = ["gptzero", "pangram", "sapling", "isgen"]
RATE_COLS = [f"{tool}_ai_rate_percent" for tool in TOOLS]
CONFLICT_RATE_RANGE_THRESHOLD = 40.0

REPORT_LINES = []


def log(message=""):
    print(message)
    REPORT_LINES.append(str(message))


def save_report():
    with open(REPORT_TXT, "w", encoding="utf-8") as file:
        file.write("\n".join(REPORT_LINES))


if not os.path.exists(INPUT_XLSX):
    raise FileNotFoundError(f"Input file not found:\n{INPUT_XLSX}")

df = pd.read_excel(INPUT_XLSX)
df.columns = [str(column).strip() for column in df.columns]

required_cols = [
    "Language",
    "University",
    "Major",
    "Assignment Type",
    "Year",
    "file_name",
    "word_count",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
]
missing_cols = [column for column in required_cols if column not in df.columns]
if missing_cols:
    raise ValueError("Missing required columns:\n" + "\n".join(missing_cols))

for column in ["Language", "University", "Major", "Assignment Type", "Year"]:
    df[column] = df[column].astype(str).str.strip()

df["word_count"] = pd.to_numeric(df["word_count"], errors="coerce")
for column in RATE_COLS:
    df[column] = pd.to_numeric(df[column], errors="coerce")


def normalize_label(value):
    if pd.isna(value):
        return np.nan

    label = str(value).strip().lower()
    if "error" in label:
        return "ERROR"
    if "mixed" in label:
        return "MIXED"
    if "human" in label and "ai" not in label:
        return "HUMAN"
    if "ai" in label:
        return "AI"
    return np.nan


for tool in TOOLS:
    df[f"{tool}_label_norm"] = df[f"{tool}_result"].apply(normalize_label)


def is_coding_assignment(value):
    assignment_type = str(value).strip().lower()
    return any(term in assignment_type for term in ("code", "coding", "program"))


df["is_coding_assignment"] = df["Assignment Type"].apply(is_coding_assignment)


def weighted_combined_ai_rate(row):
    # Coding assignments use Pangram because it is the available detector.
    if row["is_coding_assignment"]:
        return row["pangram_ai_rate_percent"]

    available_rates = {
        tool: float(row[f"{tool}_ai_rate_percent"])
        for tool in TOOLS
        if pd.notna(row[f"{tool}_ai_rate_percent"])
    }
    if not available_rates:
        return np.nan

    available_weight_sum = sum(NORM_WEIGHTS[tool] for tool in available_rates)
    return sum(
        available_rates[tool] * NORM_WEIGHTS[tool]
        for tool in available_rates
    ) / available_weight_sum


def combined_label(rate, threshold=50.0):
    if pd.isna(rate):
        return np.nan
    return "AI" if rate >= threshold else "HUMAN"


df["combined_ai_rate_percent"] = df.apply(weighted_combined_ai_rate, axis=1)
df["combined_label_50"] = df["combined_ai_rate_percent"].apply(combined_label)

log("=" * 90)
log("AI-DETECTION ANALYSIS")
log("=" * 90)
log(f"Input workbook: {INPUT_XLSX}")
log(f"Rows/files: {len(df)}")
log(f"Output folder: {OUTPUT_DIR}")
log("")
log("Combined AI-rate method:")
log("  - Non-coding assignments: accuracy-weighted detector average")
log("  - Coding assignments: Pangram only")
log("")
log("Normalized detector weights:")
for tool in TOOLS:
    log(f"  - {tool}: raw={RAW_WEIGHTS[tool]:.2f}, normalized={NORM_WEIGHTS[tool]:.4f}")
log("")


def summarize_rates(data, label):
    log("-" * 90)
    log(label)
    log("-" * 90)

    columns = {
        "Combined AI-rate": "combined_ai_rate_percent",
        "GPTZero AI-rate": "gptzero_ai_rate_percent",
        "Pangram AI-rate": "pangram_ai_rate_percent",
        "Sapling AI-rate": "sapling_ai_rate_percent",
        "Isgen AI-rate": "isgen_ai_rate_percent",
    }

    for display_name, column in columns.items():
        values = pd.to_numeric(data[column], errors="coerce").dropna()
        if values.empty:
            log(f"{display_name}: no valid values")
            continue
        log(
            f"{display_name}: n={len(values)}, mean={values.mean():.2f}, "
            f"median={values.median():.2f}, sd={values.std(ddof=1):.2f}, "
            f"min={values.min():.2f}, max={values.max():.2f}"
        )
    log("")


summarize_rates(df, "OVERALL DESCRIPTIVE RESULTS")

GROUP_COLS = ["University", "Major", "Year", "Language", "Assignment Type"]


def grouped_summary_table(data, group_col):
    table = (
        data.groupby(group_col)
        .agg(
            n_files=("file_name", "count"),
            mean_combined_ai_rate=("combined_ai_rate_percent", "mean"),
            median_combined_ai_rate=("combined_ai_rate_percent", "median"),
            sd_combined_ai_rate=("combined_ai_rate_percent", "std"),
            mean_gptzero=("gptzero_ai_rate_percent", "mean"),
            mean_pangram=("pangram_ai_rate_percent", "mean"),
            mean_sapling=("sapling_ai_rate_percent", "mean"),
            mean_isgen=("isgen_ai_rate_percent", "mean"),
            mean_word_count=("word_count", "mean"),
        )
        .reset_index()
        .sort_values(["mean_combined_ai_rate", "n_files"], ascending=[False, False])
    )

    for column in table.columns:
        if column not in [group_col, "n_files"]:
            table[column] = table[column].round(2)
    return table


group_tables = {}
for group_col in GROUP_COLS:
    table = grouped_summary_table(df, group_col)
    group_tables[group_col] = table

    log("=" * 90)
    log(f"GROUPED RESULTS BY {group_col.upper()}")
    log("=" * 90)
    log(table.to_string(index=False))
    log("")


def save_bar_plot(table, group_col, value_col, title, filename):
    plot_data = table[[group_col, value_col, "n_files"]].copy()
    plot_data = plot_data.sort_values(value_col, ascending=False)

    plt.figure(figsize=(12, 6))
    plt.bar(plot_data[group_col].astype(str), plot_data[value_col])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(value_col.replace("_", " "))
    plt.title(title)
    plt.tight_layout()

    output_path = os.path.join(PLOTS_DIR, filename)
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()
    log(f"Saved plot: {output_path}")


for group_col in GROUP_COLS:
    save_bar_plot(
        group_tables[group_col],
        group_col,
        "mean_combined_ai_rate",
        f"Mean Combined AI-rate by {group_col}",
        f"mean_combined_ai_rate_by_{group_col.lower().replace(' ', '_')}.png",
    )

SCATTER_COLS = [
    "combined_ai_rate_percent",
    "gptzero_ai_rate_percent",
    "pangram_ai_rate_percent",
    "sapling_ai_rate_percent",
    "isgen_ai_rate_percent",
]

for column in SCATTER_COLS:
    plot_data = df[["word_count", column]].dropna()
    if len(plot_data) < 3:
        continue

    plt.figure(figsize=(8, 6))
    plt.scatter(plot_data["word_count"], plot_data[column])
    plt.xlabel("word_count")
    plt.ylabel(column)
    plt.title(f"word_count vs {column}")
    plt.tight_layout()

    output_path = os.path.join(PLOTS_DIR, f"scatter_word_count_vs_{column}.png")
    plt.savefig(output_path, dpi=200, bbox_inches="tight")
    plt.close()
    log(f"Saved plot: {output_path}")

log("")


def kruskal_for_group(data, group_col, value_col="combined_ai_rate_percent", min_group_n=3):
    subset = data[[group_col, value_col]].dropna()
    group_counts = subset[group_col].value_counts()
    eligible_groups = group_counts[group_counts >= min_group_n].index
    subset = subset[subset[group_col].isin(eligible_groups)]

    groups = []
    labels = []
    for label, group in subset.groupby(group_col):
        values = group[value_col].dropna().values
        if len(values) >= min_group_n:
            groups.append(values)
            labels.append(label)

    if len(groups) < 2:
        return None, None, None

    statistic, p_value = kruskal(*groups)
    return statistic, p_value, labels


def pairwise_posthoc_kruskal(data, group_col, value_col="combined_ai_rate_percent", min_group_n=3):
    subset = data[[group_col, value_col]].dropna()
    group_counts = subset[group_col].value_counts()
    eligible_groups = group_counts[group_counts >= min_group_n].index
    subset = subset[subset[group_col].isin(eligible_groups)]

    labels = sorted(subset[group_col].unique())
    results = []

    for index, first_label in enumerate(labels):
        for second_label in labels[index + 1:]:
            first_values = subset.loc[subset[group_col] == first_label, value_col].dropna().values
            second_values = subset.loc[subset[group_col] == second_label, value_col].dropna().values

            if len(first_values) >= min_group_n and len(second_values) >= min_group_n:
                statistic, p_value = kruskal(first_values, second_values)
                results.append(
                    {
                        "group_col": group_col,
                        "group_1": first_label,
                        "group_2": second_label,
                        "n1": len(first_values),
                        "n2": len(second_values),
                        "mean1": np.mean(first_values),
                        "mean2": np.mean(second_values),
                        "statistic": statistic,
                        "p_value": p_value,
                    }
                )

    if not results:
        return pd.DataFrame()

    posthoc = pd.DataFrame(results)
    reject, adjusted_p_values, _, _ = multipletests(posthoc["p_value"], method="fdr_bh")
    posthoc["p_adj_fdr_bh"] = adjusted_p_values
    posthoc["significant"] = reject
    posthoc["mean_diff"] = posthoc["mean1"] - posthoc["mean2"]

    for column in ["mean1", "mean2", "statistic", "p_value", "p_adj_fdr_bh", "mean_diff"]:
        posthoc[column] = posthoc[column].astype(float)

    return posthoc.sort_values(
        ["significant", "p_adj_fdr_bh", "p_value"],
        ascending=[False, True, True],
    )


significant_overall = []

log("=" * 90)
log("SIGNIFICANCE TESTS: GROUP DIFFERENCES IN COMBINED AI-RATE")
log("=" * 90)

for group_col in GROUP_COLS:
    statistic, p_value, labels = kruskal_for_group(df, group_col)
    if statistic is None:
        log(f"{group_col}: not enough groups with >=3 files each")
        log("")
        continue

    log(f"{group_col}: Kruskal-Wallis H={statistic:.4f}, p={p_value:.6f}, groups_tested={len(labels)}")

    if p_value < 0.05:
        log(f"  Significant overall difference found for {group_col}")
        significant_overall.append(group_col)
        posthoc = pairwise_posthoc_kruskal(df, group_col)
        significant_pairs = posthoc[posthoc["significant"]].copy()

        if not significant_pairs.empty:
            log("  Significant pairwise differences after FDR correction:")
            display_cols = ["group_1", "group_2", "n1", "n2", "mean1", "mean2", "mean_diff", "p_adj_fdr_bh"]
            log(significant_pairs[display_cols].to_string(index=False))
        else:
            log("  No pairwise comparisons remained significant after FDR correction.")
    else:
        log(f"  No significant overall difference found for {group_col}")
    log("")

log("=" * 90)
log("RELATIONSHIP BETWEEN WORD COUNT AND AI-RATES")
log("=" * 90)

corr_results = []
for column in SCATTER_COLS:
    subset = df[["word_count", column]].dropna()
    if len(subset) < 3:
        log(f"{column}: not enough data")
        continue

    spearman_rho, spearman_p = spearmanr(subset["word_count"], subset[column])
    pearson_r, pearson_p = pearsonr(subset["word_count"], subset[column])
    corr_results.append(
        {
            "variable": column,
            "n": len(subset),
            "spearman_rho": spearman_rho,
            "spearman_p": spearman_p,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
        }
    )
    log(
        f"{column}: n={len(subset)}, Spearman rho={spearman_rho:.4f}, p={spearman_p:.6f}; "
        f"Pearson r={pearson_r:.4f}, p={pearson_p:.6f}"
    )

log("")


def available_labels(row):
    return [
        row[f"{tool}_label_norm"]
        for tool in TOOLS
        if pd.notna(row[f"{tool}_label_norm"])
        and row[f"{tool}_label_norm"] not in ["ERROR", "MIXED"]
    ]


def available_rates(row):
    return [
        float(row[f"{tool}_ai_rate_percent"])
        for tool in TOOLS
        if pd.notna(row[f"{tool}_ai_rate_percent"])
    ]


def label_conflict(row):
    return len(set(available_labels(row))) >= 2


def rate_conflict(row, threshold=CONFLICT_RATE_RANGE_THRESHOLD):
    rates = available_rates(row)
    return len(rates) >= 2 and max(rates) - min(rates) >= threshold


df["label_conflict"] = df.apply(label_conflict, axis=1)
df["rate_conflict"] = df.apply(rate_conflict, axis=1)
df["any_conflict"] = df["label_conflict"] | df["rate_conflict"]

conflict_df = df[df["any_conflict"]].copy()
conflict_df["available_tool_count"] = conflict_df.apply(lambda row: len(available_rates(row)), axis=1)
conflict_df["rate_range"] = conflict_df.apply(
    lambda row: max(available_rates(row)) - min(available_rates(row))
    if len(available_rates(row)) >= 2
    else np.nan,
    axis=1,
)

conflict_keep_cols = [
    "file_name",
    "University",
    "Major",
    "Assignment Type",
    "Year",
    "Language",
    "word_count",
    "combined_ai_rate_percent",
    "gptzero_ai_rate_percent",
    "gptzero_result",
    "pangram_ai_rate_percent",
    "pangram_result",
    "sapling_ai_rate_percent",
    "sapling_result",
    "isgen_ai_rate_percent",
    "isgen_result",
    "available_tool_count",
    "label_conflict",
    "rate_conflict",
    "rate_range",
]
conflict_df[conflict_keep_cols].to_csv(CONFLICT_CSV, index=False, encoding="utf-8-sig")

log("=" * 90)
log("CONFLICTING FILES BETWEEN TOOLS")
log("=" * 90)
log(f"Conflict rule: labels disagree or detector AI-rate range >= {CONFLICT_RATE_RANGE_THRESHOLD} points")
log(f"Number of conflicting files: {len(conflict_df)}")
log("")

if not conflict_df.empty:
    conflict_by_type = (
        conflict_df.groupby("Assignment Type")
        .size()
        .reset_index(name="n_conflicting_files")
        .sort_values("n_conflicting_files", ascending=False)
    )
    log("Conflicting files by Assignment Type:")
    log(conflict_by_type.to_string(index=False))
    log("")

    log("First 50 conflicting files:")
    display_cols = [
        "file_name",
        "Assignment Type",
        "University",
        "Major",
        "Year",
        "Language",
        "combined_ai_rate_percent",
        "label_conflict",
        "rate_conflict",
        "rate_range",
    ]
    log(conflict_df[display_cols].head(50).to_string(index=False))
    log("")
else:
    log("No conflicting files found.")
    log("")


def normalize_for_match(name):
    stem = Path(str(name)).stem
    for pattern in [
        r"_ai[_ -]?trimmed$",
        r"_trimmed$",
        r"_cleaned$",
        r"_clean$",
        r"_processed$",
        r"_ocr$",
        r"_final$",
        r"_reviewed$",
        r"_ready$",
    ]:
        stem = re.sub(pattern, "", stem, flags=re.IGNORECASE)
    return re.sub(r"[_\-\s]+", " ", stem).strip().lower()


log("=" * 90)
log("COPYING RAW PDFs FOR CONFLICTING FILES")
log("=" * 90)

pdf_index = {}
if os.path.exists(ASSIGNMENTS_HUB):
    log(f"Indexing PDFs under: {ASSIGNMENTS_HUB}")
    all_pdfs = list(Path(ASSIGNMENTS_HUB).rglob("*.pdf"))
    for pdf_path in all_pdfs:
        pdf_index.setdefault(normalize_for_match(pdf_path.name), []).append(str(pdf_path))
    log(f"Indexed {len(all_pdfs)} PDF files.")
else:
    log(f"PDF root folder not found: {ASSIGNMENTS_HUB}")
    all_pdfs = []

copied_count = 0
not_found = []
multi_found = []

if not conflict_df.empty and all_pdfs:
    for _, row in conflict_df.iterrows():
        file_name = str(row["file_name"])
        match_key = normalize_for_match(file_name)
        matches = pdf_index.get(match_key, [])

        if not matches:
            matches = [
                str(pdf_path)
                for pdf_path in all_pdfs
                if match_key in normalize_for_match(pdf_path.name)
                or normalize_for_match(pdf_path.name) in match_key
            ]

        if not matches:
            not_found.append(file_name)
            log(f"[NOT FOUND] {file_name}")
            continue

        if len(matches) > 1:
            multi_found.append((file_name, matches[:10]))
            log(f"[MULTIPLE MATCHES] {file_name} -> using all matches")

        for source in matches:
            base_name = Path(source).name
            destination = os.path.join(CONFLICT_PDFS_DIR, base_name)

            if os.path.exists(destination):
                stem = Path(base_name).stem
                suffix = Path(base_name).suffix
                copy_number = 2
                while os.path.exists(destination):
                    destination = os.path.join(CONFLICT_PDFS_DIR, f"{stem}__copy{copy_number}{suffix}")
                    copy_number += 1

            try:
                shutil.copy2(source, destination)
                copied_count += 1
                log(f"[COPIED] {file_name} -> {destination}")
            except Exception as error:
                log(f"[COPY FAILED] {file_name} -> {source} | {error}")

log("")
log(f"Total copied PDFs: {copied_count}")
log(f"TXT files with no matched PDF: {len(not_found)}")
log(f"TXT files with multiple PDF matches: {len(multi_found)}")
log("")

if not_found:
    log("TXT files with no matched PDF:")
    for file_name in not_found[:100]:
        log(f"  - {file_name}")
    log("")

if multi_found:
    log("TXT files with multiple matched PDFs:")
    for file_name, matches in multi_found[:50]:
        log(f"  - {file_name}")
        for match in matches:
            log(f"      {match}")
    log("")

log("=" * 90)
log("SHORT FINDINGS SUMMARY")
log("=" * 90)

overall_mean = df["combined_ai_rate_percent"].mean()
overall_median = df["combined_ai_rate_percent"].median()
log(f"Overall combined AI-rate: mean={overall_mean:.2f}, median={overall_median:.2f}")

for group_col in GROUP_COLS:
    table = group_tables[group_col]
    if not table.empty:
        highest = table.iloc[0]
        lowest = table.iloc[-1]
        log(
            f"{group_col}: highest mean combined AI-rate = {highest[group_col]} "
            f"({highest['mean_combined_ai_rate']:.2f}), lowest = {lowest[group_col]} "
            f"({lowest['mean_combined_ai_rate']:.2f})"
        )

if significant_overall:
    log("Significant overall group differences were found for:")
    for group_col in significant_overall:
        log(f"  - {group_col}")
else:
    log("No overall significant group differences were found at p < 0.05.")

if corr_results:
    corr_df = pd.DataFrame(corr_results)
    significant_spearman = corr_df[corr_df["spearman_p"] < 0.05]
    if not significant_spearman.empty:
        log("Significant Spearman relationships with word_count:")
        for _, result in significant_spearman.iterrows():
            log(f"  - {result['variable']}: rho={result['spearman_rho']:.4f}, p={result['spearman_p']:.6f}")
    else:
        log("No significant Spearman relationships with word_count were found at p < 0.05.")

log("")
log(f"Conflicting files CSV saved to: {CONFLICT_CSV}")
log(f"Conflicting raw PDFs folder: {CONFLICT_PDFS_DIR}")
log(f"Plots folder: {PLOTS_DIR}")
log("")
log("=" * 90)
log("DONE")
log("=" * 90)
log(f"TXT report saved to: {REPORT_TXT}")

save_report()

### 2.1 Assignment-Level Summary

The full assignments dataset used a combined AI-rate calculated from weighted detector contributions. Weights were derived from testing-sample calibration results, giving greater influence to detectors with stronger measured performance. For coding assignments, the combined score used Pangram only because the writing-focused detectors were unavailable.

Combined AI-rates varied substantially across files. Descriptive comparisons were produced by university, language, major, year, assignment type, and word count. Files with strong detector disagreement were exported to a conflict table with their raw PDFs for manual review.

## 3. Detector Conflict Analysis

This section examines disagreement among detectors rather than combined AI rates.

The next code block reloads the assignment-level detection workbook and, for each file, computes detector-score range, standard deviation, mean absolute deviation, coefficient of variation, and pairwise detector gaps. It flags files exceeding disagreement thresholds; compares detector AI rates between English and Arabic files; applies significance tests with multiple-testing correction; and saves conflict and language-comparison tables with supporting visualizations.

In [ ]:
# Install dependencies in Colab if needed.
!pip -q install openpyxl scipy statsmodels

import itertools
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import levene, mannwhitneyu, ttest_ind
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")

# Configure paths with environment variables rather than notebook-specific paths.
INPUT_XLSX = os.environ.get("AI_DETECTION_INPUT_XLSX")
OUTPUT_DIR = os.environ.get(
    "AI_DETECTION_OUTPUT_DIR",
    os.path.dirname(INPUT_XLSX) if INPUT_XLSX else ".",
)

if not INPUT_XLSX:
    raise ValueError("Set the AI_DETECTION_INPUT_XLSX environment variable to the input workbook path.")

PLOTS_DIR = os.path.join(OUTPUT_DIR, "plots")
REPORT_TXT = os.path.join(OUTPUT_DIR, "Conflict_Between_Tools_Report.txt")
PER_FILE_CSV = os.path.join(OUTPUT_DIR, "conflict_per_file_metrics.csv")
LANGUAGE_CSV = os.path.join(OUTPUT_DIR, "detector_language_comparison.csv")

LANGUAGE_COL = "Language"
FILE_COL = "file_name"
TOOLS = ["gptzero", "pangram", "sapling", "isgen"]
RATE_COLS = {tool: f"{tool}_ai_rate_percent" for tool in TOOLS}

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

REPORT_LINES = []


def log(message=""):
    print(message)
    REPORT_LINES.append(str(message))


def save_report():
    with open(REPORT_TXT, "w", encoding="utf-8") as file:
        file.write("\n".join(REPORT_LINES))


if not os.path.exists(INPUT_XLSX):
    raise FileNotFoundError(f"Workbook not found: {INPUT_XLSX}")

df = pd.read_excel(INPUT_XLSX)
df.columns = [str(column).strip() for column in df.columns]

required = [FILE_COL, LANGUAGE_COL, *RATE_COLS.values()]
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError("Missing required columns:\n" + "\n".join(missing))

for column in RATE_COLS.values():
    df[column] = pd.to_numeric(df[column], errors="coerce")

df[LANGUAGE_COL] = df[LANGUAGE_COL].astype(str).str.strip()


def normalize_language(value):
    normalized = str(value).strip().lower()
    if normalized in ["english", "eng", "en"]:
        return "English"
    if normalized in ["arabic", "arab", "ar"]:
        return "Arabic"
    return str(value).strip()


df["Language_norm"] = df[LANGUAGE_COL].apply(normalize_language)
lang_df = df[df["Language_norm"].isin(["English", "Arabic"])].copy()


def valid_rates_from_row(row):
    names, values = [], []
    for tool in TOOLS:
        value = row[RATE_COLS[tool]]
        if pd.notna(value):
            names.append(tool)
            values.append(float(value))
    return names, values


def pairwise_abs_diffs(rate_dict):
    differences = {}
    for first, second in itertools.combinations(TOOLS, 2):
        first_value = rate_dict.get(first, np.nan)
        second_value = rate_dict.get(second, np.nan)
        differences[f"absdiff_{first}_{second}"] = (
            abs(first_value - second_value)
            if pd.notna(first_value) and pd.notna(second_value)
            else np.nan
        )
    return differences


per_file_rows = []
for _, row in df.iterrows():
    rate_dict = {tool: row[RATE_COLS[tool]] for tool in TOOLS}
    _, values = valid_rates_from_row(row)
    values_array = np.array(values, dtype=float)

    n_tools = len(values_array)
    mean_rate = np.nan if n_tools == 0 else float(np.mean(values_array))
    median_rate = np.nan if n_tools == 0 else float(np.median(values_array))
    min_rate = np.nan if n_tools == 0 else float(np.min(values_array))
    max_rate = np.nan if n_tools == 0 else float(np.max(values_array))
    rate_range = np.nan if n_tools == 0 else float(max_rate - min_rate)
    sd_rate = np.nan if n_tools <= 1 else float(np.std(values_array, ddof=1))
    mad_rate = np.nan if n_tools == 0 else float(np.mean(np.abs(values_array - mean_rate)))
    cv_rate = (
        np.nan
        if n_tools <= 1 or mean_rate == 0 or pd.isna(mean_rate)
        else float(sd_rate / mean_rate)
    )

    per_file_rows.append(
        {
            FILE_COL: row[FILE_COL],
            "Language_norm": row["Language_norm"],
            "n_available_tools": n_tools,
            "mean_tool_rate": mean_rate,
            "median_tool_rate": median_rate,
            "min_tool_rate": min_rate,
            "max_tool_rate": max_rate,
            "range_tool_rate": rate_range,
            "sd_tool_rate": sd_rate,
            "mad_tool_rate": mad_rate,
            "cv_tool_rate": cv_rate,
            "conflict_ge_20": bool(pd.notna(rate_range) and rate_range >= 20),
            "conflict_ge_30": bool(pd.notna(rate_range) and rate_range >= 30),
            "conflict_ge_40": bool(pd.notna(rate_range) and rate_range >= 40),
            **pairwise_abs_diffs(rate_dict),
        }
    )

per_file_df = pd.DataFrame(per_file_rows)
per_file_df.to_csv(PER_FILE_CSV, index=False, encoding="utf-8-sig")

log("=" * 90)
log("CONFLICT BETWEEN AI TOOLS: PER-FILE METRICS")
log("=" * 90)
log(f"Files analyzed: {len(per_file_df)}")
log()

summary_stats = {
    "Mean range": per_file_df["range_tool_rate"].mean(),
    "Median range": per_file_df["range_tool_rate"].median(),
    "SD of range": per_file_df["range_tool_rate"].std(ddof=1),
    "Mean SD across tools per file": per_file_df["sd_tool_rate"].mean(),
    "Mean MAD across tools per file": per_file_df["mad_tool_rate"].mean(),
    "Mean CV across tools per file": per_file_df["cv_tool_rate"]
    .replace([np.inf, -np.inf], np.nan)
    .mean(),
    "Files with range >= 20": int(per_file_df["conflict_ge_20"].sum()),
    "Files with range >= 30": int(per_file_df["conflict_ge_30"].sum()),
    "Files with range >= 40": int(per_file_df["conflict_ge_40"].sum()),
}

for label, value in summary_stats.items():
    log(f"{label}: {value:.2f}" if isinstance(value, float) else f"{label}: {value}")
log()

pairwise_cols = [column for column in per_file_df.columns if column.startswith("absdiff_")]
pairwise_summary = (
    per_file_df[pairwise_cols]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .T.reset_index()
    .rename(columns={"index": "pair"})
)

for column in ["mean", "median", "std", "min", "max"]:
    pairwise_summary[column] = pairwise_summary[column].round(2)

log("=" * 90)
log("PAIRWISE ABSOLUTE DIFFERENCES BETWEEN DETECTORS")
log("=" * 90)
log(pairwise_summary.to_string(index=False))
log()

conflict_lang_summary = (
    per_file_df[per_file_df["Language_norm"].isin(["English", "Arabic"])]
    .groupby("Language_norm")
    .agg(
        n_files=(FILE_COL, "count"),
        mean_range=("range_tool_rate", "mean"),
        median_range=("range_tool_rate", "median"),
        mean_sd=("sd_tool_rate", "mean"),
        mean_mad=("mad_tool_rate", "mean"),
        conflict_ge_20=("conflict_ge_20", "sum"),
        conflict_ge_30=("conflict_ge_30", "sum"),
        conflict_ge_40=("conflict_ge_40", "sum"),
    )
    .reset_index()
)

for column in ["mean_range", "median_range", "mean_sd", "mean_mad"]:
    conflict_lang_summary[column] = conflict_lang_summary[column].round(2)

log("=" * 90)
log("CONFLICT METRICS BY LANGUAGE")
log("=" * 90)
log(conflict_lang_summary.to_string(index=False))
log()


def cliffs_delta(x, y):
    x = np.array(x, dtype=float)
    y = np.array(y, dtype=float)
    if len(x) == 0 or len(y) == 0:
        return np.nan

    greater = sum(np.sum(value > y) for value in x)
    less = sum(np.sum(value < y) for value in x)
    return (greater - less) / (len(x) * len(y))


language_rows = []
log("=" * 90)
log("DETECTOR AI-RATE COMPARISON: ENGLISH VS ARABIC")
log("=" * 90)

for tool in TOOLS:
    column = RATE_COLS[tool]
    english = pd.to_numeric(
        lang_df.loc[lang_df["Language_norm"] == "English", column], errors="coerce"
    ).dropna()
    arabic = pd.to_numeric(
        lang_df.loc[lang_df["Language_norm"] == "Arabic", column], errors="coerce"
    ).dropna()

    if len(english) == 0 or len(arabic) == 0:
        log(f"{tool.upper()}: not enough English/Arabic data")
        continue

    # Mann-Whitney U is the primary comparison; the t-test is retained as a reference.
    u_statistic, mannwhitney_p = mannwhitneyu(english, arabic, alternative="two-sided")
    _, levene_p = levene(english, arabic, center="median")
    _, ttest_p = ttest_ind(
        english,
        arabic,
        equal_var=(levene_p >= 0.05),
        nan_policy="omit",
    )

    language_rows.append(
        {
            "tool": tool,
            "english_n": len(english),
            "arabic_n": len(arabic),
            "english_mean": english.mean(),
            "arabic_mean": arabic.mean(),
            "english_median": english.median(),
            "arabic_median": arabic.median(),
            "mean_diff_english_minus_arabic": english.mean() - arabic.mean(),
            "mannwhitney_u": u_statistic,
            "mannwhitney_p": mannwhitney_p,
            "ttest_p": ttest_p,
            "cliffs_delta": cliffs_delta(english, arabic),
        }
    )

language_comp_df = pd.DataFrame(language_rows)

if len(language_comp_df):
    reject, adjusted_p, _, _ = multipletests(
        language_comp_df["mannwhitney_p"], method="fdr_bh"
    )
    language_comp_df["mannwhitney_p_adj_fdr"] = adjusted_p
    language_comp_df["significant_after_fdr"] = reject

    numeric_columns = [
        "english_mean",
        "arabic_mean",
        "english_median",
        "arabic_median",
        "mean_diff_english_minus_arabic",
        "mannwhitney_u",
        "mannwhitney_p",
        "ttest_p",
        "cliffs_delta",
        "mannwhitney_p_adj_fdr",
    ]
    for column in numeric_columns:
        language_comp_df[column] = pd.to_numeric(
            language_comp_df[column], errors="coerce"
        ).round(4)

language_comp_df.to_csv(LANGUAGE_CSV, index=False, encoding="utf-8-sig")

if len(language_comp_df):
    log(language_comp_df.to_string(index=False))
log()

plt.figure(figsize=(8, 5))
per_file_df["range_tool_rate"].dropna().hist(bins=30)
plt.xlabel("Per-file range across detectors")
plt.ylabel("Number of files")
plt.title("Distribution of detector conflict (range across tools)")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "hist_range_tool_rate.png"), dpi=200, bbox_inches="tight")
plt.close()

plt.figure(figsize=(10, 5))
plt.bar(pairwise_summary["pair"], pairwise_summary["mean"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Mean absolute difference")
plt.title("Average disagreement between detector pairs")
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, "bar_pairwise_mean_absdiff.png"), dpi=200, bbox_inches="tight")
plt.close()

if len(language_comp_df):
    plot_df = language_comp_df[["tool", "english_mean", "arabic_mean"]].copy()
    positions = np.arange(len(plot_df))
    width = 0.35

    plt.figure(figsize=(9, 5))
    plt.bar(positions - width / 2, plot_df["english_mean"], width=width, label="English")
    plt.bar(positions + width / 2, plot_df["arabic_mean"], width=width, label="Arabic")
    plt.xticks(positions, plot_df["tool"].str.upper())
    plt.ylabel("Mean AI-rate")
    plt.title("Detector mean AI-rate in English vs Arabic files")
    plt.legend()
    plt.tight_layout()
    plt.savefig(
        os.path.join(PLOTS_DIR, "bar_detector_means_english_vs_arabic.png"),
        dpi=200,
        bbox_inches="tight",
    )
    plt.close()

if len(conflict_lang_summary):
    plt.figure(figsize=(7, 5))
    plt.bar(conflict_lang_summary["Language_norm"], conflict_lang_summary["mean_range"])
    plt.ylabel("Mean per-file range")
    plt.title("Average detector conflict by language")
    plt.tight_layout()
    plt.savefig(os.path.join(PLOTS_DIR, "bar_mean_range_by_language.png"), dpi=200, bbox_inches="tight")
    plt.close()

log("Saved plots to:")
log(f"  {PLOTS_DIR}")
log()

log("=" * 90)
log("SHORT INTERPRETATION")
log("=" * 90)

worst_pair = pairwise_summary.sort_values("mean", ascending=False).iloc[0]
best_pair = pairwise_summary.sort_values("mean", ascending=True).iloc[0]

log(
    f"The detector pair with the largest average disagreement was {worst_pair['pair']} "
    f"(mean absolute difference = {worst_pair['mean']:.2f})."
)
log(
    f"The detector pair with the smallest average disagreement was {best_pair['pair']} "
    f"(mean absolute difference = {best_pair['mean']:.2f})."
)
log(
    f"Across all files, {int(per_file_df['conflict_ge_40'].sum())} files had very high conflict "
    f"(range >= 40 points), {int(per_file_df['conflict_ge_30'].sum())} had range >= 30, "
    f"and {int(per_file_df['conflict_ge_20'].sum())} had range >= 20."
)

if len(language_comp_df):
    significant = language_comp_df[language_comp_df["significant_after_fdr"]]
    if len(significant):
        log("After FDR correction, significant English-vs-Arabic detector differences were found for:")
        for _, result in significant.iterrows():
            direction = (
                "English > Arabic"
                if result["mean_diff_english_minus_arabic"] > 0
                else "Arabic > English"
            )
            log(
                f"  - {result['tool'].upper()}: English mean={result['english_mean']:.2f}, "
                f"Arabic mean={result['arabic_mean']:.2f}, "
                f"adjusted p={result['mannwhitney_p_adj_fdr']:.4f} ({direction})"
            )
    else:
        log("After FDR correction, no detector showed a statistically significant English-vs-Arabic difference.")
log()

log("=" * 90)
log("DONE")
log("=" * 90)
log(f"TXT report saved to: {REPORT_TXT}")
log(f"Per-file conflict CSV saved to: {PER_FILE_CSV}")
log(f"Language comparison CSV saved to: {LANGUAGE_CSV}")
save_report()

### 3.1 Conflict Analysis

Detector disagreement was substantial: many files had wide ranges between the highest and lowest AI-rate estimates, and many met the script’s high-conflict thresholds. A single-detector interpretation would therefore be less stable.

Pairwise comparisons identify detector pairs with greater divergence. The workflow uses a weighted combined score and manual review for conflicting files. English–Arabic comparisons assess whether detector behavior differs by assignment language. Detector disagreement should be documented explicitly in the broader workflow.

## Notebook scope and prerequisites

Read the sections in this order:

1. **Testing-sample calibration:** detector performance measurement.
2. **Assignment-level summary:** dataset description and combined AI-rate calculation.
3. **Conflict analysis:** detector disagreements requiring additional caution.

This notebook requires outputs from the earlier preprocessing pipeline:

- trimmed, standardized assignment-text files;
- AI-detection API results stored in Excel workbooks; and
- configuration paths matching the current Google Drive structure.

If these inputs or paths change, update the configuration blocks at the start of the relevant code sections.